#read table from bronze layer

In [0]:
df = spark.read.format("delta").table("`ttc-lakehouse`.bronze.calendar")

##flag standardization (0/1 -> Boolean)

In [0]:
from pyspark.sql.functions import when, col

days = ["sunday", "saturday", "friday", "thursday", "wednesday", "tuesday", "monday"]

for day in days:
    df = df.withColumn(day, when(col(day) == 1, True).when(col(day) == 0, False).otherwise(col(day)))

##Data type standardization : cast start_date , end_date from String to Date

In [0]:
from pyspark.sql.functions import to_date, regexp_replace, col

df = df.withColumn("start_date", to_date(regexp_replace(col("start_date"), r"(\d{4})(\d{2})(\d{2})", r"$1-$2-$3"), "yyyy-MM-dd")) \
       .withColumn("end_date", to_date(regexp_replace(col("end_date"), r"(\d{4})(\d{2})(\d{2})", r"$1-$2-$3"), "yyyy-MM-dd"))

##write into silver layer

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("`ttc-lakehouse`.silver.calendar")